In [ ]:
import onnx
from collections import Counter
import onnx.shape_inference as si

ONNX_PATH = "/home/dos/.cache/ivit/models/efficientnet-b0.onnx"

m = onnx.load(ONNX_PATH) 
nodes = m.graph.node 
print(f"total nodes: {len(nodes)}")

ops = Counter(n.op_type for n in nodes)
print("top ops:", dict(ops.most_common(10)))
print()

for x in m.graph.input:
    dims = [d.dim_value or d.dim_param or "?" for d in x.type.tensor_type.shape.dim] 
    print(f"INPUT{x.name}: {dims}")

for x in m.graph.output: 
    dims = [d.dim_value or d.dim_param or "?"  for d in x.type.tensor_type.shape.dim] 
    print(f"OUTPUT {x.name}: {dims}")
    print()

m2 = si.infer_shapes(m)
shape_map = {} 
for vi in list(m2.graph.value_info) + list(m2.graph.input) + list(m2.graph.output):
    shape_map[vi.name] = [ 
        d.dim_value if d.dim_value > 0 else None
        for d in vi.type.tensor_type.shape.dim ]
    
print("=== GlobalAveragePool / Reduce-style pools ===")
pool_types = ("GlobalAveragePool", "AveragePool", "ReduceMean")
for n in nodes:
    if n.op_type in pool_types:
        inp = shape_map.get(n.input[0])
        out = shape_map.get(n.output[0]) 
        print(f"[{n.op_type}] {n.name}\nin={inp}out={out}")